In [0]:
%sql
-- Create 'trend_market_project' folder
CREATE VOLUME IF NOT EXISTS msbabigdata.spark.trend_market_project;

PART 1: Initialize Paths and Ingest Data

In [0]:
# Step 1: Define paths
trips_path = "/Volumes/msbabigdata/spark/trend_market_project/yellow_tripdata_2023_sample.parquet"
zone_path = "/Volumes/msbabigdata/spark/trend_market_project/Taxi_zone_lookup.csv"

In [0]:
# Step 2: Ingest data
df_raw = spark.read.parquet(trips_path)
zone_df = spark.read.option("header", "true").option("inferSchema", "true").csv(zone_path)

In [0]:
# Step 3: Standardize column names to lowercase immediately (fixes the 'Airport_fee' issue)
df_trips = df_raw.toDF(*[c.lower() for c in df_raw.columns])

In [0]:
# --- VALIDATION OF PART 1: Profile the raw data ---
print(f"Raw record count: {df_trips.count()}")
df_trips.select("tpep_pickup_datetime", "fare_amount", "trip_distance").summary().show()

Raw record count: 3000
+-------+------------------+-----------------+
|summary|       fare_amount|    trip_distance|
+-------+------------------+-----------------+
|  count|              3000|             3000|
|   mean|19.197499999999984| 3.85141333333333|
| stddev|19.531142412174486|4.717278570128284|
|    min|             -80.0|              0.0|
|    25%|               8.6|             1.21|
|    50%|              12.8|              2.2|
|    75%|              22.6|              4.4|
|    max|             400.0|            75.57|
+-------+------------------+-----------------+



PART 2: Build the Cleaning Pipeline

In [0]:
from pyspark.sql import functions as F

In [0]:
def clean_taxi_data(df):
    return (df
        # 1. Handle Erroneous Dates: Filter strictly for 2023
        .filter(F.year("tpep_pickup_datetime") == 2023)
        
        # 2. Handle Null/Invalid Zone IDs (NYC zones are 1-263)
        .filter(F.col("pulocationid").isNotNull() & (F.col("pulocationid").between(1, 263)))
        .filter(F.col("dolocationid").isNotNull() & (F.col("dolocationid").between(1, 263)))
        
        # 3. Handle Outliers: Fare >= $2.50 (NYC min) and sensible distance
        .filter((F.col("fare_amount") >= 2.5) & (F.col("fare_amount") < 500))
        .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
        
        # 4. Remove Duplicate records
        .dropDuplicates()
        
        # 5. Timestamp formatting & Timezone Normalization
        # We explicitly convert the UTC data to New York time
        .withColumn("tpep_pickup_datetime", F.from_utc_timestamp(F.col("tpep_pickup_datetime"), "America/New_York"))
        .withColumn("tpep_dropoff_datetime", F.from_utc_timestamp(F.col("tpep_dropoff_datetime"), "America/New_York"))
        
        # Add a date column for partitioning later
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    )

In [0]:
df_cleaned = clean_taxi_data(df_trips)

In [0]:
# --- VALIDATION of PART 2: Check Cleaning Results ---
print(f"Cleaned record count: {df_cleaned.count()}")

Cleaned record count: 2852


In [0]:
# Check for remaining nulls in critical columns
df_cleaned.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ["pulocationid", "fare_amount"]]).show()

+------------+-----------+
|pulocationid|fare_amount|
+------------+-----------+
|           0|          0|
+------------+-----------+



In [0]:
# Verify Date Range (Should only be 2023)
df_cleaned.select(F.min("tpep_pickup_datetime"), F.max("tpep_pickup_datetime")).show()

+-------------------------+-------------------------+
|min(tpep_pickup_datetime)|max(tpep_pickup_datetime)|
+-------------------------+-------------------------+
|      2022-12-31 19:00:47|      2023-02-28 20:03:01|
+-------------------------+-------------------------+



PART 3: Join with Zone Lookup

In [0]:
# Join cleaned trips with the lookup table
# Use left join to ensure we keep trip records even if a zone ID is weird

# Join 1: Attach names for the Pickup Location
final_df = df_cleaned.join(
    zone_df.select(F.col("LocationID").alias("puloc_id"), 
                   F.col("Borough").alias("pickup_borough"), 
                   F.col("Zone").alias("pickup_zone")), 
    df_cleaned.pulocationid == F.col("puloc_id"), 
    "left"
).drop("puloc_id")

In [0]:
# Join 2: Attach names for the Dropoff Location (Requirement B)
final_df = final_df.join(
    zone_df.select(F.col("LocationID").alias("doloc_id"), 
                   F.col("Borough").alias("dropoff_borough"), 
                   F.col("Zone").alias("dropoff_zone")), 
    final_df.dolocationid == F.col("doloc_id"), 
    "left"
).drop("doloc_id")

In [0]:
# --- VALIDATION of PART 3: Check Join ---
# Ensure we have borough names instead of just IDs
final_df.select("pulocationid", "pickup_borough", "pickup_zone").distinct().limit(5).show()

+------------+--------------+-------------------+
|pulocationid|pickup_borough|        pickup_zone|
+------------+--------------+-------------------+
|          79|     Manhattan|       East Village|
|         144|     Manhattan|Little Italy/NoLiTa|
|          75|     Manhattan|  East Harlem South|
|         194|     Manhattan|    Randalls Island|
|         140|     Manhattan|    Lenox Hill East|
+------------+--------------+-------------------+



In [0]:
# --- VALIDATION of PART 3: check both ends of the trip
final_df.select(
    "pulocationid", "pickup_borough", 
    "dolocationid", "dropoff_borough"
).distinct().limit(5).show()

+------------+--------------+------------+---------------+
|pulocationid|pickup_borough|dolocationid|dropoff_borough|
+------------+--------------+------------+---------------+
|          43|     Manhattan|         237|      Manhattan|
|         164|     Manhattan|         236|      Manhattan|
|         164|     Manhattan|         232|      Manhattan|
|          13|     Manhattan|         211|      Manhattan|
|         239|     Manhattan|         133|       Brooklyn|
+------------+--------------+------------+---------------+



PART 4: Write to Delta Lake

In [0]:
target_table = "msbabigdata.spark.trend_market_cleaned_trips"

(final_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .partitionBy("pickup_date", "pulocationid")
 .saveAsTable(target_table))

In [0]:
# --- VALIDATION of PART 4: Query the Delta Table ---
print("Final Table Sample:")
display(spark.sql(f"SELECT * FROM {target_table} LIMIT 10"))

Final Table Sample:


vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pickup_date,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone
2,2022-12-31T19:52:15.000Z,2022-12-31T20:01:38.000Z,1.0,1.69,1.0,N,231,158,1,11.4,1.0,0.5,2.0,0.0,1.0,18.4,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Meatpacking/West Village West
2,2022-12-31T19:44:16.000Z,2022-12-31T19:55:47.000Z,2.0,2.15,1.0,N,231,164,2,13.5,1.0,0.5,0.0,0.0,1.0,18.5,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Midtown South
2,2022-12-31T19:23:15.000Z,2022-12-31T19:59:20.000Z,1.0,6.14,1.0,N,231,238,1,38.7,1.0,0.5,8.74,0.0,1.0,52.44,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Upper West Side North
2,2022-12-31T19:32:17.000Z,2022-12-31T20:25:08.000Z,1.0,9.43,1.0,Y,231,116,1,56.9,1.0,0.5,0.0,0.0,1.0,61.9,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Hamilton Heights
1,2022-12-31T19:45:24.000Z,2022-12-31T20:07:35.000Z,3.0,3.7,1.0,N,231,161,2,21.2,3.5,0.5,0.0,0.0,1.0,26.2,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Midtown Center
1,2022-12-31T19:54:44.000Z,2022-12-31T20:13:06.000Z,1.0,5.9,1.0,N,231,161,3,26.1,3.5,0.5,0.0,0.0,1.0,31.1,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Midtown Center
1,2022-12-31T19:48:00.000Z,2022-12-31T20:28:58.000Z,1.0,5.8,1.0,N,231,239,1,38.7,3.5,0.5,6.56,0.0,1.0,50.26,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Upper West Side South
1,2022-12-31T19:41:30.000Z,2022-12-31T20:03:23.000Z,2.0,2.8,1.0,N,231,164,1,22.6,3.5,0.5,5.0,0.0,1.0,32.6,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Midtown South
1,2022-12-31T19:14:15.000Z,2022-12-31T19:22:39.000Z,0.0,1.6,1.0,N,231,114,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Greenwich Village South
1,2022-12-31T19:18:42.000Z,2022-12-31T19:34:09.000Z,1.0,2.1,1.0,N,231,107,1,15.6,3.5,0.5,4.1,0.0,1.0,24.7,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Gramercy


In [0]:
# =============================================================================
# MEMBER 2 — Demand Aggregation & Scoring
# Hengrui Li
# Input:  msbabigdata.spark.trend_market_cleaned_trips  (Member 1 output)
# Output: msbabigdata.spark.trend_market_demand_scores
# =============================================================================
# NOTE ON TIMESTAMPS:
# The cleaned table timestamps have been converted from UTC to NYC time by
# Member 1 (from_utc_timestamp). Because the raw sample data was already in
# NYC local time, the timestamps in the cleaned table are shifted ~5 hours
# earlier than actual pickup times. This does not affect the logic or
# correctness of the pipeline — when the full TLC dataset is loaded, Member 1
# should remove the timezone conversion step, and this notebook requires
# no changes.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
# =============================================================================
# PART 1: Load cleaned trips from Member 1's Delta table
# =============================================================================
 
source_table = "msbabigdata.spark.trend_market_cleaned_trips"
df = spark.table(source_table)
 
# Only select the 4 columns we need — everything else is irrelevant for demand scoring
df = df.select("pulocationid", "pickup_zone", "pickup_borough", "tpep_pickup_datetime")
 
# --- VALIDATION ---
print("=== PART 1: Input Data ===")
print(f"Row count: {df.count()}")
df.show(5)

=== PART 1: Input Data ===
Row count: 2852
+------------+--------------------+--------------+--------------------+
|pulocationid|         pickup_zone|pickup_borough|tpep_pickup_datetime|
+------------+--------------------+--------------+--------------------+
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:52:15|
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:44:16|
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:23:15|
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:32:17|
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:45:24|
+------------+--------------------+--------------+--------------------+
only showing top 5 rows


In [0]:
# =============================================================================
# PART 2: Feature Extraction — time_bucket + day_type
# =============================================================================
# TIME BUCKETS (4 segments with business meaning):
#   morning_rush : 06–10  commute to work
#   midday       : 10–16  off-peak daytime
#   evening_rush : 16–20  commute home + dining
#   night        : 20–06  late night / overnight
#
# DAY TYPE (2 categories):
#   weekday  : Monday–Friday  (dayofweek 2–6 in Spark, where 1=Sunday)
#   weekend  : Saturday–Sunday (dayofweek 1 or 7)
 
def assign_time_bucket(hour_col):
    return (
        F.when((hour_col >= 6)  & (hour_col < 10), "morning_rush")
         .when((hour_col >= 10) & (hour_col < 16), "midday")
         .when((hour_col >= 16) & (hour_col < 20), "evening_rush")
         .otherwise("night")
    )
 
def assign_day_type(dow_col):
    # Spark dayofweek: 1=Sunday, 2=Monday, ..., 7=Saturday
    return F.when(dow_col.isin(1, 7), "weekend").otherwise("weekday")
 
df_features = (
    df
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow",  F.dayofweek("tpep_pickup_datetime"))
    .withColumn("time_bucket", assign_time_bucket(F.col("hour")))
    .withColumn("day_type",    assign_day_type(F.col("dow")))
)
 
# --- VALIDATION ---
print("\n=== PART 2: Feature Extraction ===")
print("Time bucket distribution:")
df_features.groupBy("time_bucket").count().orderBy("count", ascending=False).show()
print("Day type distribution:")
df_features.groupBy("day_type").count().orderBy("count", ascending=False).show()


=== PART 2: Feature Extraction ===
Time bucket distribution:
+------------+-----+
| time_bucket|count|
+------------+-----+
|evening_rush| 2846|
|       night|    6|
+------------+-----+

Day type distribution:
+--------+-----+
|day_type|count|
+--------+-----+
| weekday| 1889|
| weekend|  963|
+--------+-----+



In [0]:
# =============================================================================
# PART 3: Zone Aggregation — trip_count per (zone × time_bucket × day_type)
# =============================================================================
 
df_agg = (
    df_features
    .groupBy("pulocationid", "pickup_zone", "pickup_borough", "time_bucket", "day_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.countDistinct(F.to_date("tpep_pickup_datetime")).alias("days_observed")
    )
)
 
# --- VALIDATION ---
print("\n=== PART 3: Zone Aggregation ===")
print(f"Aggregated rows (zone × time_bucket × day_type cells): {df_agg.count()}")
print("Sample — top zones by trip count:")
df_agg.orderBy("trip_count", ascending=False).show(10)
 


=== PART 3: Zone Aggregation ===
Aggregated rows (zone × time_bucket × day_type cells): 143
Sample — top zones by trip count:
+------------+--------------------+--------------+------------+--------+----------+-------------+
|pulocationid|         pickup_zone|pickup_borough| time_bucket|day_type|trip_count|days_observed|
+------------+--------------------+--------------+------------+--------+----------+-------------+
|         132|         JFK Airport|        Queens|evening_rush| weekday|       164|            2|
|         230|Times Sq/Theatre ...|     Manhattan|evening_rush| weekday|        98|            2|
|          79|        East Village|     Manhattan|evening_rush| weekday|        88|            2|
|         249|        West Village|     Manhattan|evening_rush| weekday|        81|            2|
|         114|Greenwich Village...|     Manhattan|evening_rush| weekday|        78|            2|
|         234|            Union Sq|     Manhattan|evening_rush| weekday|        76|      

In [0]:
# =============================================================================
# PART 4: Demand Score Computation
# =============================================================================
# demand_score = trip_count / avg(trip_count across all zones for same time window)
#
# score > 1.0  →  above average demand for this time window
# score = 1.5  →  50% above average
# score < 1.0  →  below average
#
# We use a Window partitioned by (time_bucket, day_type) so the comparison is
# always apples-to-apples: a zone is only compared against other zones in the
# same time window, not against the global average.
 
window_spec = Window.partitionBy("time_bucket", "day_type")
 
df_scored = (
    df_agg
    .withColumn("avg_trip_count", F.avg("trip_count").over(window_spec))
    .withColumn("stddev_trip_count", F.stddev("trip_count").over(window_spec))
    .withColumn(
        "demand_score",
        F.round(F.col("trip_count") / F.col("avg_trip_count"), 4)
    )
)
 
# --- VALIDATION ---
print("\n=== PART 4: Demand Scores ===")
print("Score distribution summary:")
df_scored.select("demand_score").summary().show()
print("Top 10 highest-scoring zones:")
df_scored.orderBy("demand_score", ascending=False).show(10)


=== PART 4: Demand Scores ===
Score distribution summary:
+-------+------------------+
|summary|      demand_score|
+-------+------------------+
|  count|               143|
|   mean|0.9999951048951061|
| stddev|1.0516073723971762|
|    min|            0.0403|
|    25%|             0.127|
|    50%|            0.7617|
|    75%|            1.5234|
|    max|            6.6122|
+-------+------------------+

Top 10 highest-scoring zones:
+------------+--------------------+--------------+------------+--------+----------+-------------+------------------+------------------+------------+
|pulocationid|         pickup_zone|pickup_borough| time_bucket|day_type|trip_count|days_observed|    avg_trip_count| stddev_trip_count|demand_score|
+------------+--------------------+--------------+------------+--------+----------+-------------+------------------+------------------+------------+
|         132|         JFK Airport|        Queens|evening_rush| weekday|       164|            2| 24.80263157894737

In [0]:
 
# =============================================================================
# PART 5: Low-Confidence Flagging
# =============================================================================
# A zone-time cell is flagged as low_confidence if ANY of:
#   (a) trip_count < 3      — too few trips, pattern not reliable
#   (b) days_observed < 2   — only seen on 1 day, could be an anomaly
#   (c) CoV > 2.0           — coefficient of variation too high (stddev / mean),
#                             demand is too erratic to trust
#                             CoV = stddev / avg across zones in same window
#
# NOTE ON THRESHOLDS:
# These thresholds are intentionally low because the sample data is small.
# With full TLC data (millions of rows), raise trip_count threshold to ~20
# and days_observed to ~10 for more conservative filtering.
 
TRIP_COUNT_MIN  = 3
DAYS_OBSERVED_MIN = 2
COV_MAX = 2.0
 
df_final = (
    df_scored
    .withColumn(
        "coeff_of_variation",
        F.round(
            F.when(
                F.col("avg_trip_count") > 0,
                F.col("stddev_trip_count") / F.col("avg_trip_count")
            ).otherwise(None),
            4
        )
    )
    .withColumn(
        "is_low_confidence",
        (F.col("trip_count") < TRIP_COUNT_MIN) |
        (F.col("days_observed") < DAYS_OBSERVED_MIN) |
        (F.col("coeff_of_variation") > COV_MAX)
    )
    # Clean up intermediate columns not needed downstream
    .drop("stddev_trip_count")
)
 
# --- VALIDATION ---
print("\n=== PART 5: Low-Confidence Flagging ===")
flag_counts = df_final.groupBy("is_low_confidence").count()
flag_counts.show()
 
total = df_final.count()
low_conf = df_final.filter(F.col("is_low_confidence") == True).count()
print(f"Low-confidence cells: {low_conf} / {total} ({low_conf/total*100:.1f}%)")
print(f"High-confidence cells available for ranking: {total - low_conf}")
 
print("\nSample of flagged zones:")
df_final.filter(F.col("is_low_confidence") == True) \
        .select("pickup_zone", "time_bucket", "day_type", "trip_count", "days_observed", "demand_score", "is_low_confidence") \
        .show(5)
 
print("\nSample of high-confidence zones:")
df_final.filter(F.col("is_low_confidence") == False) \
        .orderBy("demand_score", ascending=False) \
        .select("pickup_zone", "pickup_borough", "time_bucket", "day_type", "trip_count", "demand_score", "is_low_confidence") \
        .show(10)


=== PART 5: Low-Confidence Flagging ===
+-----------------+-----+
|is_low_confidence|count|
+-----------------+-----+
|            false|   52|
|             true|   91|
+-----------------+-----+

Low-confidence cells: 91 / 143 (63.6%)
High-confidence cells available for ranking: 52

Sample of flagged zones:
+--------------------+------------+--------+----------+-------------+------------+-----------------+
|         pickup_zone| time_bucket|day_type|trip_count|days_observed|demand_score|is_low_confidence|
+--------------------+------------+--------+----------+-------------+------------+-----------------+
|Central Harlem North|evening_rush| weekday|         1|            1|      0.0403|             true|
|       South Jamaica|evening_rush| weekday|         1|            1|      0.0403|             true|
|  Claremont/Bathgate|evening_rush| weekday|         1|            1|      0.0403|             true|
|       East New York|evening_rush| weekday|         1|            1|      0.0403| 

In [0]:
# =============================================================================
# PART 6: Write to Delta Lake
# =============================================================================
# Output schema for Member 3 (Heuristic Ranking):
#   pulocationid        — zone ID (join key)
#   pickup_zone         — human-readable zone name
#   pickup_borough      — borough name
#   time_bucket         — morning_rush / midday / evening_rush / night
#   day_type            — weekday / weekend
#   trip_count          — raw pickup count in this cell
#   days_observed       — number of distinct days this cell was observed
#   avg_trip_count      — citywide average for this time window
#   demand_score        — trip_count / avg_trip_count (>1.0 = above average)
#   coeff_of_variation  — stddev/mean across zones in window (signal reliability)
#   is_low_confidence   — boolean flag for Member 3 to filter out
 
target_table = "msbabigdata.spark.trend_market_demand_scores"
 
(
    df_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)
 
# --- VALIDATION ---
print("\n=== PART 6: Delta Lake Write Validation ===")
df_check = spark.table(target_table)
print(f"Rows written to {target_table}: {df_check.count()}")
print("\nFinal table schema:")
df_check.printSchema()
print("\nFinal table sample (top demand zones, high confidence only):")
display(
    spark.sql(f"""
        SELECT pickup_zone, pickup_borough, time_bucket, day_type,
               trip_count, demand_score, is_low_confidence
        FROM {target_table}
        WHERE is_low_confidence = false
        ORDER BY demand_score DESC
        LIMIT 15
    """)
)
 


=== PART 6: Delta Lake Write Validation ===
Rows written to msbabigdata.spark.trend_market_demand_scores: 143

Final table schema:
root
 |-- pulocationid: long (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- time_bucket: string (nullable = true)
 |-- day_type: string (nullable = true)
 |-- trip_count: long (nullable = true)
 |-- days_observed: long (nullable = true)
 |-- avg_trip_count: double (nullable = true)
 |-- demand_score: double (nullable = true)
 |-- coeff_of_variation: double (nullable = true)
 |-- is_low_confidence: boolean (nullable = true)


Final table sample (top demand zones, high confidence only):


pickup_zone,pickup_borough,time_bucket,day_type,trip_count,demand_score,is_low_confidence
JFK Airport,Queens,evening_rush,weekday,164,6.6122,false
Times Sq/Theatre District,Manhattan,evening_rush,weekday,98,3.9512,false
East Village,Manhattan,evening_rush,weekday,88,3.548,false
West Village,Manhattan,evening_rush,weekday,81,3.2658,false
Greenwich Village South,Manhattan,evening_rush,weekday,78,3.1448,false
Union Sq,Manhattan,evening_rush,weekday,76,3.0642,false
Clinton East,Manhattan,evening_rush,weekday,74,2.9836,false
LaGuardia Airport,Queens,evening_rush,weekday,73,2.9432,false
Meatpacking/West Village West,Manhattan,evening_rush,weekday,62,2.4997,false
Midtown East,Manhattan,evening_rush,weekday,60,2.4191,false
